In [1]:
import os 
import phoenix as px
from dotenv import load_dotenv
px.launch_app(port=6006)

/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


❗️ The launch_app `port` parameter is deprecated and will be removed in a future release. Use the `PHOENIX_PORT` environment variable instead.


/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/opt/homebrew/Cellar/python@3.12/3.12.13/Frameworks/Python.framework/Versions/3.12/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [2]:
from phoenix.otel import register 
# os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "http://localhost:6006"
tracer_provider=register(auto_instrument=True, project_name='python-phoenix-pra')
"The register function from phoenix.otel sets up instrumentation so your agent will automatically sends traces to Phoenix."
#here we will grab the tracer obj to manually instrument some functions later no

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: python-phoenix-pra
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



'The register function from phoenix.otel sets up instrumentation so your agent will automatically sends traces to Phoenix.'

In [3]:
from opentelemetry import trace
tracer=trace.get_tracer(__name__)

In this section, we’ll build our travel agent. Users will be able to describe their destination, travel dates, and interests, and the agent will generate a customized, budget-conscious itinerary.

Tavily Search to help the tools gather general information about each destination and Open-Mateo to get the weather

In [4]:
#httpx """is a fast, specialized Go-based command-line utility used by security professionals to probe and analyze large lists of HTTP/HTTPS targets."""

In [5]:
# print("TAVILY:", os.getenv("TAVILY_API_KEY"))
import time

In [6]:
import httpx 

@tracer.chain(name='search-api') #IT tracks a function as step in llm pipeline # manual tracing wrapper
def _search_api(query:str):
    api_key=os.getenv('TAVILY_API_KEY')

    resp=httpx.post(
        "https://api.tavily.com/search",
        json={
            'api_key':api_key,
            'query':query,
            'max_results':3,
            'search_depth':'advanced',
            'include_answer':True
        },
        timeout=8
    )
    if resp.status_code!=200:
        print('Travil api faild',resp.text)
        return None

    data=resp.json()

    answer=data.get('answer','') or ''
    print('answer',answer)

    snippets=[item.get('content','') for item in data.get('results',[])]

    combined=' '.join([answer]+snippets).strip()
    return combined[:400] if combined else None


@tracer.chain(name='weather-api')
def _weather(dest):
    try:
        g = httpx.get(f'https://geocoding-api.open-meteo.com/v1/search?name={dest}', timeout=5)

        if g.status_code != 200 or not g.json().get('results'):
            return "Location not found"

        lat = g.json()['results'][0]['latitude']
        lon = g.json()['results'][0]['longitude']

        for _ in range(3):  # retry loop
            try:
                w = httpx.get(
                    f'https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true',
                    timeout=5
                ).json()

                cw = w.get('current_weather')
                if cw:
                    return f"Weather now: {cw.get('temperature')}°C, wind {cw.get('windspeed')} km/h."

            except Exception as e:
                print("Retrying weather API...", e)
                time.sleep(1)

        return "Weather data unavailable"

    except Exception as e:
        print("Weather API completely failed:", e)
        return "Weather service error"


Our agent will have access to three tools:

Essential Info – Provides key travel details about the destination, such as weather and general conditions.

Budget Basics – Offers insights into travel costs and helps plan budgets based on selected activities.

Local Flavor – Recommends unique local experiences and cultural highlights.

In [7]:
from agno.tools import tool 

@tool 
def essential_info(destination:str):
    """Get essental info using search and weather apis"""

    parts=[]
    q=f'{destination} travel essential weather best time top attractions etiquette'
    s=_search_api(q)
    if s:
        parts.append(f'{destination} essentials: {s}')

    else:
        parts.append(f'{destination} is popular travel destination .Expect local culture, cuisine, and landmarks worth ')

    weather=_weather(destination)
    if weather:
        parts.append(weather)

    return f"{destination} essentials:\n"+"\n".join(parts)

In [8]:
@tool
def budget_basics(destination:str,duration:str)->str:
    """summarize traval cost categories"""

    q=f'{destination} travel budget average daily costs {duration}'
    s=_search_api(q)
    if s:
        return f"{destination} budget ({duration}):{s}"
    return f"Budget for {duration} in {destination} depends on lodging, meals, transport, and attractions."

@tool
def local_flavor(destination: str, interests: str = "local culture")->str:
    """Suggest authentic local experiences"""
    q=f'{destination} authentic local experiance {interests}'
    s=_search_api(q)
    if s:
        return f"{destination} {interests}: {s}"
    return f"Explore {destination}'s unique {interests} through markets, neighborhoods, and local eateries."




Build the Agent
Next, we’ll construct our agent. The Agno framework makes this process straightforward by allowing us to easily define key parameters such as the model, instructions, and tools.

In [9]:
from agno.agent import Agent 
from agno.models.groq import Groq

In [10]:
model=Groq(id='llama-3.1-8b-instant',
    api_key=os.environ['GROQ_API_KEY']
)

In [23]:
trip_agent=Agent(
    name='make my trip',
    role='AI travel agent',
    model=model,
    instructions=(
        "You are a friendly and knowledgeable travel planner. "
        "Combine multiple tools to create a trip plan including essentials, budget, and local flavor. "
        "Keep the tone natural, clear, and under 1000 words."
    ),
    markdown=True,
    tools=[essential_info,budget_basics,local_flavor]
)

In [24]:
destination='akola'
duration='5 days'
interests='food'

query=f"""
Plan a {duration} trip to {destination}. Focus on {interests} Include essential info,budget breakdown and local experiences."""

In [25]:
def agent_task(query):
    trip_agent.print_response(query,stream=True)


agent_task(query)

/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Define Dataset to Run on the Agent
In order to experiment with our agent, we first need to define a dataset for it to run on. This provides a standardized way to evaluate the agent’s behavior across consistent inputs.

In this example, we’ll use a small dataset of ten examples and upload it to Phoenix using the Phoenix Client. Once uploaded, all experiments will be tracked alongside this dataset.

In [26]:
from phoenix.client import Client 
import pandas as pd

client=Client() # reading / interacting with traces

In [27]:
# client.get_spans()

In [29]:
queries = [
    "Plan a 7-day trip to Italy focused on art, history, and local food. Include essential travel info, a budget estimate, and key attractions in Rome, Florence, and Venice.",
    "Create a 4-day itinerary for Seoul centered on K-pop, fashion districts, and street food. Include transportation tips and a mid-range budget.",
    "Plan a romantic 5-day getaway to Paris with emphasis on museums, wine tasting, and scenic walks. Provide cost estimates and essential travel notes.",
    "Design a 3-day budget trip to Mexico City focusing on food markets, archaeological sites, and nightlife. Include daily cost breakdowns.",
    "Prepare a 6-day itinerary for New Zealand’s South Island with a focus on outdoor adventure, hikes, and photography spots. Include travel logistics and gear essentials.",
    "Plan a 10-day trip across Spain, hitting Barcelona, Madrid, and Seville. Focus on architecture, tapas, and cultural festivals. Include a detailed budget.",
    "Create a 5-day family-friendly itinerary for Singapore with theme parks, nature activities, and kid-friendly dining. Include entry fees and transit costs.",
    "Plan a 4-day luxury spa and relaxation trip to Bali. Include premium resorts, wellness activities, and a high-end budget.",
    "Design a 7-day solo backpacking trip through Thailand with hostels, street food, and cultural attractions. Provide safety essentials and budget breakdown.",
    "Create a 3-day weekend itinerary for New York City focusing on art galleries, rooftop restaurants, and iconic attractions. Include estimated costs.",
]



dataset_df = pd.DataFrame(data={"input": queries})

dataset = client.datasets.create_dataset(
    dataframe=dataset_df, name="travel-questions", input_keys=["input"]
)

DatasetUploadError: Dataset upload failed: Dataset with the same name already exists: name='travel-questions'

Define Evaluators
Next, we need a way to assess the agent’s outputs. This is where Evals come in. Evals provide a structured method for measuring whether an agent’s responses meet the requirements defined in a dataset—such as accuracy, relevance, consistency, or safety.

In this tutorial, we will be using LLM-as-a-Judge Evals, which rely on another LLM acting as the evaluator. We define a prompt describing the exact criteria we want to evaluate, and then pass the input and the agent-generated output from each dataset example into that evaluator prompt. The evaluator LLM then returns a score along with a natural-language explanation justifying why the score was assigned.

This allows us to automatically grade the agent’s performance across many examples, giving us quantitative metrics as well as qualitative insight into failure cases.

In [30]:
ANSWER_RELEVANCE_PROMPT_TEMPLATE = """
You will be given a travel-planning query and an itinerary answer. Your task is to decide whether
the answer correctly follows the user's instructions. An answer is "incorrect" if it contradicts,
ignores, or fails to include required elements from the query (such as trip length, destination,
themes, budget details, or essential info). It is also "incorrect" if it adds irrelevant or
contradictory details.

    [BEGIN DATA]
    ************
    [Query]: {{input}}
    ************
    [Answer]: {{output}}
    ************
    [END DATA]

Explain step-by-step how you determined your judgment. Then provide a final LABEL:
- Use "correct" if the answer follows the query accurately and fully.
- Use "incorrect" if it deviates from the query or omits required information.

Your final output must be only one word: "correct" or "incorrect".
"""


fter defining the evaluator prompt, we use the Phoenix Evals library to construct an evaluator instance. Notice that the evaluator LLM is separate from the model powering the agent itself.

In [31]:
from phoenix.evals import create_classifier 
from phoenix.evals.llm import LLM

llm = LLM(
    provider="litellm",
    model="groq/llama-3.1-8b-instant"
)

relevancy_evaluator=create_classifier(
    name='Answer relevance',
    llm=llm,
    prompt_template=ANSWER_RELEVANCE_PROMPT_TEMPLATE,
    choices={'correct':1.0,'incorrect':0.0}
)

In [32]:
BUDGET_CONSISTENCY_PROMPT_TEMPLATE = """
You will be given a travel-planning query and an itinerary answer. Your task is to determine whether
the answer provides a consistent and mathematically coherent budget. An answer is "incorrect" if:
- The summed minimum costs of all listed budget categories exceed the stated minimum total estimate.
- The summed maximum costs of all listed budget categories exceed the stated maximum total estimate.
- The total estimate claims a range that cannot be derived from (or contradicted by) the itemized ranges.
- The answer lists budget items but provides a total that is not numerically aligned with them.
- The answer contradicts itself regarding pricing or cost ranges.

    [BEGIN DATA]
    ************
    [Query]: {{input}}
    ************
    [Answer]: {{output}}
    ************
    [END DATA]

Explain step-by-step how you evaluated the itemized costs and the final total, including whether
the ranges mathematically match. Then provide a final LABEL:
- Use "correct" if the budget totals are consistent with the itemized values.
- Use "incorrect" if the totals contradict or cannot be derived from the itemized values.

Your final output must be only one word: "correct" or "incorrect".
"""

In [33]:
llm = LLM(
    provider="litellm",
    model="groq/llama-3.3-70b-versatile"
)
budget_evaluator=create_classifier(
    name='BUDGET CONSISTENCY',
    llm=llm,
    prompt_template=BUDGET_CONSISTENCY_PROMPT_TEMPLATE,
    choices={'correct':1.0,'incorrect':0.0}
)

Run Experiment
The last step before running our experiment is to explicitly define the task. Although we know we are evaluating an agent, we must wrap simple function that takes an input and returns the agent’s output. This function becomes the task that the experiment will execute for each example in the dataset.

In [34]:
def agent_task(input):
    query=input['input']
    response=trip_agent.run(query,stream=False)
    return response.content

After defining the task, we construct our experiment by providing the dataset, the evaluators, and any relevant metadata. Once everything is configured, we can run the experiment.

Depending on the size of the dataset, complexity of the task, and the number of evaluators, the run may take a few minutes to complete.

In [35]:
from phoenix.client.experiments import run_experiment

experiment=run_experiment(
    dataset=dataset,
    task=agent_task,
    experiment_name='inital_run',
    evaluators=[relevancy_evaluator,budget_evaluator]
)

🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDox/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDoz


answer Best time to visit Italy is spring (March to May) for fewer crowds and pleasant weather; top attractions include Tuscany, Amalfi Coast, and Sicily; Italian etiquette includes greeting with a smile and saying "buongiorno."
answer A one-week trip to Italy typically costs around $1,650 for one person, including accommodation, food, transport, and sightseeing. For two people, it averages $3,300. A mid-range 7-day trip costs about $2,900 per person.
answer Italy offers authentic local experiences through traditional street food, regional specialties, and local markets. Local tours provide insights into Italian culture and cuisine. Seek recommendations from locals for the best food experiences.


answer Best time to visit Paris is spring (April) or fall (September-October). Top attractions include museums like Musée d’Orsay and the Louvre. French etiquette includes polite greetings and dining manners.
answer For a 5-day trip to Paris, daily costs range from $110 to $320, with transport, food, and attractions adding to the budget. Budget travelers may spend around $1,000-$1,800 per person, while mid-range budgets typically range from $2,900 for a solo traveler to $4,500 for a couple.
answer For authentic local wine tasting in Paris, visit Le Vin qui Parle or Les Caves du Louvre, both offering personalized experiences with expert sommeliers.


answer For a 3-day budget trip to Mexico City, plan for $40-60 per day, covering hostels, street food, Metro, selective attractions, and emergencies. Average daily costs range from $75-225, depending on accommodation and activities.
answer Best time to visit Mexico City is March to June; top attractions include museums and historical sites; respect local customs and language basics.


ERROR    Error calling Groq API: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust 
         your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code':         
         'tool_use_failed', 'failed_generation': 'It seems the previous search didn\'t return the requested        
         information. I will try again using different search                                                      
         parameters.\n<function=essential_info>{"destination":"Mexico City"</function>\n\n'}}

...


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



running experiment evaluations |████████▌ | 17/20 (85.0%) | ⏳ 00:43<00:09 |  3.16s/it

Traceback (most recent call last):
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/__init__.py", line 1911, in _run_single_evaluation_sync
    result = evaluator.evaluate(
             ^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/evaluators.py", line 253, in evaluate
    scores = evaluator.evaluate(kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 561, in evaluate
    return super().evaluate(eval_input=eval_input, input_mapping=input_mapping)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 413, in evaluate
    return 

running experiment evaluations |██████████| 20/20 (100.0%) | ⏳ 00:49<00:00 |  2.29s/it

Experiment completed: 10 task runs, 2 evaluator runs, 13 evaluations


Analyze Experiment Traces & Results
Now that the experiment has run on each dataset example, you can explore the results in Phoenix. You’ll be able to:

View the full trace emitted by the agent and step through each action it took

Inspect evaluation outputs for every example, including scores, labels, and explanations

Examine the evaluation traces themselves (i.e., the LLM-as-a-Judge reasoning)

Review aggregate evaluation scores across the entire dataset

Human Annotations
In addition to Evals, you can also add human annotations to your traces. These allow you to capture strong feedback, flag problematic outputs, highlight exemplary responses, and record insights that automated evaluators may miss. Human annotations get saved as part of the trace, helping you guide future iterations of your application or agent.

Iterate and Re-Run Experiment
Now that we've spent time analyzing the experiment results, it's time to iterate.

We will update the agent’s main prompt to be more intentional about budget calculations, since that area received a lower eval score. You can modify the prompt or make any other adjustments you believe will improve the agent’s performance, and then re-run the experiment to see how the outputs improve—or where they may regress.

Iteration is a key part of refining agent behavior, and each experiment provides valuable feedback to guide the next step. Once you reach eval scores and outputs that meet your expectations, you can confidently push those changes to production.

In [36]:
#--main agent with updated instructions

trip_agent = Agent(
    name="TripPlanner",
    role="AI Travel Assistant",
    model=model,
    instructions=(
        "You are a friendly and knowledgeable travel planner. "
        "Combine multiple tools to create a trip plan including essentials, budget, and local flavor. "
        "Keep the tone natural, clear, and under 1000 words. "
        "When providing budget details: Ensure the final total budget range is mathematically consistent with the sum of the itemized ranges."
    ),
    markdown=True,
    tools=[essential_info, budget_basics, local_flavor],
)
     

In [37]:
experiment = run_experiment(
    dataset=dataset,
    task=agent_task,
    experiment_name="updated agent prompt to improve budget",
    evaluators=[relevancy_evaluator, budget_evaluator],
)

🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDox/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDo0


answer Best time to visit Italy is spring (March to May) for fewer crowds and pleasant weather; top attractions include Tuscany, Amalfi Coast, and Sicily; Italian etiquette includes greeting with a smile and saying "buongiorno."
answer Italy offers authentic art, history, and local food experiences through guided tours and workshops with local artisans. Explore Renaissance art in Florence and wine tasting in Montepulciano. Enjoy traditional Italian cooking classes.
answer A one-week trip to Italy typically costs around $1,650 for one person, including accommodation, food, transport, and sightseeing. For two people, it averages $3,300. A mid-range 7-day trip costs about $2,900 per person.


answer For a 4-day trip to Seoul, plan a daily budget of $80-$130, covering accommodation, meals, and transport. Budget travelers can manage with $50-$100 per day. Luxury stays cost up to $635 per day.
answer For a 4-day trip to Seoul, plan a daily budget of $80-$130, covering accommodation, meals, and transport. Budget travelers can manage with $50-$100 per day. Luxury stays cost up to $635 per day.
answer Seoul offers K-pop concerts, street food tours, and fashion shopping in Myeongdong and Hongdae. Try traditional Korean food at Gwangjang Market. Participate in K-culture classes for dance, makeup, and cooking.
answer Best time to visit Seoul is spring (March to May) or fall (October). Top attractions include Gyeongbokgung Palace and N Seoul Tower. Essential etiquette: no tipping or blowing noses in public.


answer Best time to visit Paris is spring (April) or fall (September-October). Top attractions include museums like Musée d’Orsay and the Louvre. French etiquette includes polite greetings and dining manners.
answer For a 5-day trip to Paris, daily costs range from $110 to $320, with transport, food, and attractions adding to the budget. Budget travelers may spend around $1,000-$1,800 per person, while mid-range budgets typically range from $2,900 for a solo traveler to $4,500 for a couple.
answer For authentic local experiences in Paris, try Paris Wine Walks for tours and tastings, and explore art museums and historical walking tours in Le Marais.


answer Best time to visit Mexico City is March or November for optimal weather and lower costs. Top attractions include museums and historical sites. Basic Spanish helps navigate local areas.
answer For a 3-day budget trip to Mexico City, plan for $40-60 per day, covering hostels, street food, Metro, selective attractions, and emergencies. Average daily costs range from $75-225, depending on accommodation and activities.
answer Mexico City offers vibrant food markets, rich archaeological sites, and a lively nightlife. Visit Mercado de San Juan and explore Templo Mayor. Experience diverse nightlife at Mama Rumba and El Plaza Condesa.


...

Traceback (most recent call last):
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/__init__.py", line 1911, in _run_single_evaluation_sync
    result = evaluator.evaluate(
             ^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/evaluators.py", line 253, in evaluate
    scores = evaluator.evaluate(kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 561, in evaluate
    return super().evaluate(eval_input=eval_input, input_mapping=input_mapping)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 413, in evaluate
    return 


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

Traceback (most recent call last):
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/litellm/llms/custom_httpx/llm_http_handler.py", line 235, in _make_common_sync_call
    response = sync_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/litellm/llms/custom_httpx/http_handler.py", line 1047, in post
    raise e
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/litellm/llms/custom_httpx/http_handler.py", line 1029, in post
    response.raise_for_status()
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/httpx/_models.py", line 829, in raise_for_status


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



Traceback (most recent call last):
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/__init__.py", line 1911, in _run_single_evaluation_sync
    result = evaluator.evaluate(
             ^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/evaluators.py", line 253, in evaluate
    scores = evaluator.evaluate(kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 561, in evaluate
    return super().evaluate(eval_input=eval_input, input_mapping=input_mapping)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 413, in evaluate
    return 

Experiment completed: 10 task runs, 2 evaluator runs, 11 evaluations
